# PART 7-1. 감성 분석 - NLP 기초 (Tokenizer + Embedding)

## 수업 목표
- 리뷰 텍스트를 숫자로 변환하고 딥러닝으로 긍정/부정을 예측합니다.
- Tokenizer, Sequence, Padding, Embedding의 개념을 이해합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 문장(텍스트)을 AI가 이해할 수 있는 숫자로 바꾸는 과정입니다. |
| 코드 관점 | Tokenizer → pad_sequences → Sequential 모델 순서를 익힙니다. |
| AI 관점 | 딥러닝은 숫자만 처리할 수 있으므로 문장을 반드시 변환해야 합니다. |
| 강사 메모 | test 데이터는 절대 증폭/수정하지 않는다는 원칙을 강조하세요. |

---


## 📌 이 파트에서 사용하는 API 흐름 (Keras)

| 단계 | 역할 |
|---|---|
| ① Define | 모델 정의 (레이어 쌓기) |
| ② Compile | 손실함수 · 옵티마이저 설정 |
| ③ Fit | 학습 |
| ④ Evaluate | 평가 |
| ⑤ Predict | 예측 |


## 흐름
```
데이터 준비 → 결측치 제거 → 라벨 생성(긍정/부정)
→ train/test 분리 → train 균형 맞추기
→ Tokenizer 학습 → Sequence 변환 → Padding
→ 딥러닝 모델 학습 → 평가 → 저장
```

## 핵심 개념

| 개념 | 설명 |
|---|---|
| Tokenizer | 문장을 단어 단위로 나누고 각 단어에 번호를 붙임 |
| Sequence | 문장을 번호 배열로 변환 |
| Padding | 모든 문장 길이를 동일하게 맞춤 (짧으면 0 채우기) |
| Embedding | 번호를 의미 있는 벡터로 변환 |
| OOV | 단어집에 없는 단어 처리 토큰 |

In [1]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 셀 2. 라이브러리 불러오기

import os
import pickle
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.utils import resample          # 데이터 개수 조정(복원 추출)
from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, GlobalAveragePooling1D, Dense, Dropout
)

base_path = "/content/drive/MyDrive/Olist"
model_path = f"{base_path}/model"
os.makedirs(model_path, exist_ok=True)

In [3]:
# 셀 3. 데이터 불러오기

df = pd.read_csv(f"{base_path}/data/olist_master_data.csv")

# 감성분석에 필요한 컬럼만 선택
review_df = df[["review_comment_message", "review_score"]].copy()
print(f"전체 리뷰 수: {len(review_df):,}")

전체 리뷰 수: 119,143


In [4]:
# 셀 4. 데이터 전처리

# 1. 리뷰 내용이 없는 행 제거
review_df = review_df.dropna()
print(f"결측치 제거 후: {len(review_df):,}")

# 2. 중립(3점) 제거 → 긍정/부정 이분법 학습을 위해
review_df = review_df[review_df["review_score"] != 3]
print(f"3점 제거 후: {len(review_df):,}")

# 3. 긍정/부정 라벨 생성
#    4~5점 → 긍정(1),  1~2점 → 부정(0)
review_df["label"] = review_df["review_score"].apply(
    lambda score: 1 if score >= 4 else 0
)

print("\n라벨 분포:")
print(review_df["label"].value_counts())
# ▶ 긍정이 부정보다 약 2배 많습니다 → 불균형 처리 필요

결측치 제거 후: 50,245
3점 제거 후: 45,714

라벨 분포:
label
1    30782
0    14932
Name: count, dtype: int64


In [5]:
# 셀 5. train / test 분리
# 중요: test 데이터는 시험지 역할 → 절대 수정하지 않습니다.

X = review_df["review_comment_message"]
y = review_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # 긍정/부정 비율을 train/test에 동일하게 유지
)

print(f"train 크기: {len(X_train):,}")
print(f"test 크기:  {len(X_test):,}")

train 크기: 36,571
test 크기:  9,143


In [6]:
# 셀 6. train 데이터 균형 맞추기 (긍정:부정 = 1:1)
# test는 건드리지 않습니다!

train_df = pd.DataFrame({"text": X_train, "label": y_train})

positive_df = train_df[train_df["label"] == 1]
negative_df = train_df[train_df["label"] == 0]

print(f"균형 전 → 긍정: {len(positive_df):,}, 부정: {len(negative_df):,}")

# 부정 리뷰를 긍정 개수만큼 복원 추출(resample)
negative_upsampled = resample(
    negative_df,
    replace=True,
    n_samples=len(positive_df),
    random_state=42
)

train_balanced = pd.concat([positive_df, negative_upsampled])
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

X_train = train_balanced["text"]
y_train = train_balanced["label"]

print(f"균형 후 → {train_balanced['label'].value_counts().to_dict()}")

균형 전 → 긍정: 24,625, 부정: 11,946
균형 후 → {0: 24625, 1: 24625}


In [7]:
# 셀 7. Tokenizer - 단어집 만들기
# 각 단어에 번호를 붙이는 과정입니다.
# 예) "I love it" → {"I":23, "love":5, "it":17}

MAX_WORDS = 10000  # 자주 나오는 단어 10,000개만 사용
MAX_LEN   = 100    # 문장 길이를 100개 단어로 고정

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"  # 단어집에 없는 단어를 <OOV>로 처리
)

# train 데이터로만 단어집 학습 (test는 절대 사용 금지)
tokenizer.fit_on_texts(X_train)

word_index = tokenizer.word_index
print(f"학습된 단어 수: {len(word_index):,}")
print("자주 나오는 단어 TOP 10:")
for word, idx in list(word_index.items())[:10]:
    print(f"  {word}: {idx}")

학습된 단어 수: 13,215
자주 나오는 단어 TOP 10:
  <OOV>: 1
  o: 2
  e: 3
  produto: 4
  não: 5
  a: 6
  de: 7
  do: 8
  que: 9
  recebi: 10


In [8]:
# 셀 8. 문장 → 숫자 배열(Sequence)로 변환

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# 변환 예시 확인
print("원본 문장:", X_train.iloc[0])
print("숫자 변환:", X_train_seq[0])

원본 문장: Eu ainda não recebi o produto
숫자 변환: [35, 32, 5, 10, 2, 4]


In [9]:
# 셀 9. Padding - 문장 길이 통일
# 딥러닝은 입력 길이가 모두 같아야 합니다.
# 짧은 문장은 뒤에 0을 채우고(padding), 긴 문장은 뒤를 잘라냅니다(truncating).

X_train_pad = pad_sequences(
    X_train_seq, maxlen=MAX_LEN,
    padding="post",    # 뒤에 0 채우기
    truncating="post"  # 뒤에서 자르기
)

X_test_pad = pad_sequences(
    X_test_seq, maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print(f"train 패딩 결과: {X_train_pad.shape}")
# ▶ (학습 데이터 수, 100) → 모든 문장이 100개 숫자로 통일됩니다.

train 패딩 결과: (49250, 100)



### 🔵 ① Define → 모델 정의


In [10]:
# 셀 10. 딥러닝 모델 구성

model = Sequential([
    # Embedding: 단어 번호를 64차원 벡터로 변환 (단어의 의미를 표현)
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),

    # GlobalAveragePooling1D: 100개 단어 벡터를 하나로 요약
    GlobalAveragePooling1D(),

    # Dropout: 학습 중 랜덤하게 일부 뉴런을 끔 → 과적합 방지
    Dropout(0.3),

    # Dense: 완전연결층 (relu = 음수는 0으로 만드는 함수)
    Dense(64, activation="relu"),
    Dropout(0.3),

    # 출력층: sigmoid = 0~1 확률값 출력 (0.5 이상이면 긍정)
    Dense(1, activation="sigmoid"),
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

? 와 (unbuilt) 는 아직 입력 데이터 shape가 확정되지 않아 모델이 완전히 생성(build)되지 않았다는 의미이며, 학습하면 자동으로 해결됩니다.


### 🔵 ② Compile → 모델 컴파일


In [11]:
# 셀 11. 모델 컴파일

model.compile(
    optimizer="adam",           # 학습률 자동 조정 옵티마이저
    loss="binary_crossentropy", # 이진 분류용 손실 함수
    metrics=["accuracy"]        # 평가 지표: 정확도
)


### 🔵 ③ Fit → 학습


In [12]:
# 셀 12. 모델 학습 (약 45초)

history = model.fit(
    X_train_pad, y_train,
    epochs=5,             # 전체 데이터를 5번 반복 학습
    batch_size=64,        # 한 번에 64개씩 처리
    validation_split=0.2  # train의 20%를 검증용으로 사용
)
# ▶ val_accuracy가 accuracy와 비슷하게 올라가면 정상 학습입니다.

Epoch 1/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.8076 - loss: 0.4230 - val_accuracy: 0.9090 - val_loss: 0.2551
Epoch 2/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9103 - loss: 0.2437 - val_accuracy: 0.9090 - val_loss: 0.2331
Epoch 3/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9261 - loss: 0.2086 - val_accuracy: 0.9332 - val_loss: 0.1912
Epoch 4/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9335 - loss: 0.1901 - val_accuracy: 0.9272 - val_loss: 0.2094
Epoch 5/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9391 - loss: 0.1750 - val_accuracy: 0.9324 - val_loss: 0.1938



### 🔵 ④ Evaluate → 평가


In [13]:
# 셀 13. 모델 평가

test_loss, test_acc = model.evaluate(X_test_pad, y_test)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

286/286 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9031 - loss: 0.2510
Test Loss:     0.2510
Test Accuracy: 0.9031



### 🔵 ⑤ Predict → 예측


In [14]:
# 셀 14. 임의 문장으로 예측 테스트

def predict_sentiment(text):
    """문장이 긍정인지 부정인지 예측합니다."""
    seq    = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob   = model.predict(padded, verbose=0)[0][0]
    result = "긍정 😊" if prob >= 0.5 else "부정 😞"
    return result, round(float(prob), 3)

print(predict_sentiment("produto muito bom gostei"))  # 좋은 제품이에요
print(predict_sentiment("produto ruim nao gostei"))   # 나쁜 제품이에요
print(predict_sentiment("배송이 빠르고 상품이 좋아요"))
# ▶ 한국어는 train 데이터에 없으므로 OOV 처리 → 정확도 낮을 수 있습니다.

('긍정 😊', 0.958)
('부정 😞', 0.04)
('긍정 😊', 0.993)


In [15]:
# 셀 15. 모델과 Tokenizer 저장

# 딥러닝 모델 저장
model.save(f"{model_path}/part7_sentiment_model.h5")

# Tokenizer 저장 (단어집 정보가 담겨있음)
with open(f"{model_path}/part7_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("감성분석 모델 저장 완료!")
print("저장 위치:", model_path)

감성분석 모델 저장 완료!
저장 위치: /content/drive/MyDrive/Olist/model


---

## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 문장 → 숫자 → 딥러닝으로 긍정/부정을 예측합니다. |
| test 원칙 | test 데이터는 절대 수정하지 않습니다. (시험지 역할) |
| 다음 단계 | PART 7-2에서 사전학습된 BERT 모델을 사용합니다. |